# Passive compliance — glissando

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.path.join('../..'))
plt.style.use(os.path.join('../..', 'plot_config.mplstyle'))

OUTPUT_DIR = os.path.join('outputs', 'piano_glissando')

# Pull constants from the run script (single source of truth)
_ns = {}
with open('piano_glissando.py') as _f:
    for _line in _f:
        if _line.startswith(('N_RUNS', 'GLISSANDO_DISTANCE', 'GLISSANDO_SPEED')):
            exec(_line, _ns)
N_RUNS             = int(_ns['N_RUNS'])
GLISSANDO_DISTANCE = float(_ns['GLISSANDO_DISTANCE'])
GLISSANDO_SPEED    = float(_ns['GLISSANDO_SPEED'])
GLISSANDO_DURATION = GLISSANDO_DISTANCE / GLISSANDO_SPEED

# Software motor indices (motor_config.py)
IDX_MCP, IDX_PIP   = 7,  8
MID_MCP, MID_PIP   = 9, 10

N_GRID = 200

def load_slide(run):
    path = os.path.join(OUTPUT_DIR, f'run_{run}.csv')
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    return df[df['phase'] == 'slide_forward'].reset_index(drop=True)

def interp_trace(arr, n=N_GRID):
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, n)
    return np.interp(x_new, x_old, arr)

runs = [load_slide(r) for r in range(1, N_RUNS + 1)]
runs = [df for df in runs if df is not None and len(df) > 5]
T = np.linspace(0, GLISSANDO_DURATION, N_GRID)


## MCP joint angles during the slide

In [ ]:
# ── Plot 1: MCP joint angles during the glissando slide ──────────────────────
# Index and middle MCP angles across all runs (mean ± std).
# Shows passive compliance-driven adaptation as fingers encounter keys.

fig, ax = plt.subplots()

for color, col, label in [
    ('#0072B2', f'q_{IDX_MCP}', 'Index MCP'),
    ('#D55E00', f'q_{MID_MCP}', 'Middle MCP'),
]:
    if not runs:
        continue
    traces = np.stack([interp_trace(np.rad2deg(df[col].to_numpy())) for df in runs])
    mu, sigma = traces.mean(0), traces.std(0)
    ax.plot(T, mu, color=color, label=label)
    ax.fill_between(T, mu - sigma, mu + sigma, color=color, alpha=0.15)

ax.set_xlabel('Time [s]')
ax.set_ylabel('Motor angle [deg]')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'glissando_angles.pdf'), bbox_inches='tight')
plt.show()


## MCP torques during the slide

In [ ]:
# ── Plot 2: MCP torques during the glissando slide ───────────────────────────
# Index and middle MCP motor torques (mean ± std across runs).
# Oscillations mark key-by-key contact events.

fig, ax = plt.subplots()

for color, col, label in [
    ('#0072B2', f'tau_{IDX_MCP}', 'Index MCP'),
    ('#D55E00', f'tau_{MID_MCP}', 'Middle MCP'),
]:
    if not runs:
        continue
    traces = np.stack([interp_trace(df[col].to_numpy()) for df in runs])
    mu, sigma = traces.mean(0), traces.std(0)
    ax.plot(T, mu, color=color, label=label)
    ax.fill_between(T, mu - sigma, mu + sigma, color=color, alpha=0.15)

ax.set_xlabel('Time [s]')
ax.set_ylabel('Motor torque [N·m]')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'glissando_torques.pdf'), bbox_inches='tight')
plt.show()
